In [ ]:
import os
import json
import asyncio
from pathlib import Path
import sys, time
import subprocess
import time
import requests
from openai import OpenAI

# --- CONFIGURATION ---
REPO_PATH = "./your-repo"
OUTPUT_DIR = Path("output_analysis")
MODEL_NAME = "qwen3-vl:8b"
MAX_CHUNK_CHARS = 12000 # Roughly 3k-4k tokens

# Initialize Directories
STAGES = ["01_scout", "02_aggregate", "03_draft", "04_critique", "05_refined", "06_visual"]
for stage in STAGES:
    (OUTPUT_DIR / stage).mkdir(parents=True, exist_ok=True)

def save_json(stage, filename, data):
    try:
        with open(OUTPUT_DIR / stage / filename, 'w') as f:
            json.dump(data, f, indent=4)
    except Exception as e:
        print(f"{e}: data")

def load_json(stage, filename):
    with open(OUTPUT_DIR / stage / filename, 'r') as f:
        return json.load(f)

In [7]:
# list ollama models
!ollama list

NAME                                          ID              SIZE      MODIFIED     
qwen3-vl:8b                                   901cae732162    6.1 GB    9 days ago      
froehnerel/gorilla-openfunctions:v2-q5_K_M    d2f688262ecd    4.9 GB    2 weeks ago     
deepseek-r1:8b                                6995872bfe4c    5.2 GB    2 weeks ago     
tinyllama:latest                              2644915ede35    637 MB    4 months ago    


In [8]:
# list cached models (on mac)
!ls -F ~/.cache/huggingface/hub/

models--mlx-community--Qwen2.5-7B-Instruct-4bit/


In [9]:
# For macs, mlx-lm
# Default running a quantized version
def start_mlx_server(model_id="mlx-community/Qwen2.5-7B-Instruct-4bit", port=8080):
    # 1. Check if server is already running
    try:
        requests.get(f"http://localhost:{port}/v1/models", timeout=2)
        print(f"Server already running on port {port}.")
        return None
    except:
        print(f"Starting MLX server with {model_id}...")
        # 2. Start the process
        cmd = ["python", "-m", "mlx_lm.server", "--model", model_id, "--port", str(port)]
        process = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        
        # 3. Wait for readiness
        for _ in range(30):
            try:
                requests.get(f"http://localhost:{port}/v1/models", timeout=1)
                print("Server is ready!")
                return process
            except:
                time.sleep(2)
        print("Failed to start server.")
        return None

# Usage
server_proc = start_mlx_server()

# Connect
client = OpenAI(base_url='http://localhost:8080/v1', api_key='local')

Server already running on port 8080.


In [10]:
response = requests.get(f"http://localhost:8080/v1/models", timeout=1).json()

In [11]:
model_id = response['data'][0]['id']

In [7]:
'''
# If you want to test your server/API
with client.chat.completions.create(
    model=model_id,  # Ensure you have run 'ollama pull llama3'
    messages=[{"role": "user", "content": "Explain why the sky is blue."}],
    stream=True,
) as response:
    for chunk in response:
        content = chunk.choices[0].delta.content
        if content:
            print(content, end="", flush=True)'''

'\n# If you want to test your server/API\nwith client.chat.completions.create(\n    model=model_id,  # Ensure you have run \'ollama pull llama3\'\n    messages=[{"role": "user", "content": "Explain why the sky is blue."}],\n    stream=True,\n) as response:\n    for chunk in response:\n        content = chunk.choices[0].delta.content\n        if content:\n            print(content, end="", flush=True)'

In [ ]:
import os
import yaml

# NOTE Very different depending on the setup... 50k for a 4b quantized 8B model on an m4, ~500k for gpt-4o...
class ScoutAgent:
    def __init__(self, repo_path, extensions=None, max_chars_per_chunk=50000, debug = False, model=MODEL_NAME, stage=STAGES[0]):
        self.repo_path = repo_path
        self.extensions = extensions or ['.py', '.java', '.c', '.cpp', '.go']
        self.max_chars_per_chunk = max_chars_per_chunk
        self.debug = debug
        self.model = model
        self.stage = stage
        self.system_prompt = """
        You are a Technical Scout. Analyze the provided code files.
        Output a Markdown response with a YAML frontmatter header.
        Identify:
        1. Module Name (based on folder or primary file)
        2. Key Dependencies (imports/includes)
        3. Primary Responsibilities
        4. Public APIs/Interfaces
        """
        self.agent_prompt = """Please analyze the following code and provide a summary in Markdown format. 
        Include sections for:
        1. High-level purpose
        2. Key functions/classes
        3. Dependencies used"""

    def _should_include(self, filename):
        return any(filename.endswith(ext) for ext in self.extensions)

    def collect_files(self):
        """Walks the repo and gathers content of relevant files."""
        all_files = []
        for root, _, files in os.walk(self.repo_path):
            # Skip hidden folders like .git
            if '/.' in root: continue 

            for file in files:
                if self._should_include(file):
                    full_path = os.path.join(root, file)
                    try:
                        with open(full_path, 'r', encoding='utf-8') as f:
                            content = f.read()
                            all_files.append({
                                "path": os.path.relpath(full_path, self.repo_path),
                                "content": content
                            })
                    except Exception as e:
                        print(f"Could not read {full_path}: {e}")
        return all_files

    def create_chunks(self, all_files):
        """Groups files into chunks that fit within LLM limits."""
        chunks = []
        current_chunk = []
        current_size = 0

        for file_data in all_files:
            file_size = len(file_data['content'])
            # If a single file is huge, we might need to truncate or split it
            if current_size + file_size > self.max_chars_per_chunk and current_chunk:
                chunks.append(current_chunk)
                current_chunk = []
                current_size = 0
            
            current_chunk.append(file_data)
            current_size += file_size
        
        if current_chunk:
            chunks.append(current_chunk)
        return chunks

    # NOTE make asynch def if calling asynchronously
    def summarize_chunk(self, chunk, llm_client):
        """Sends a chunk to the LLM to get a structured summary."""
        # Format the chunk for the prompt
        formatted_content = f"{self.agent_prompt}\n\n".join([
            f"--- FILE: {f['path']} ---\n{f['content']}" 
            for f in chunk
        ])
        
        # NOTE Call await here if using asynchio
        response = llm_client.chat.completions.create(
            model=self.model, # or your preferred model
            messages=[
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": formatted_content}
            ]
        )
        return response.choices[0].message.content

    def process_all(self, chunks, llm_client):
        start = time.time()
        processed_sum = []
        def save_md(stage, filename, data):
            try:
                with open(OUTPUT_DIR / stage / filename, 'w') as f:
                    f.write(data)
            except Exception as e:
                print(f"{e}: data")
        for i, chunk in enumerate(chunks):
            print(f"Processing chunk {i}, with {len(chunk)} files")
            summary = self.summarize_chunk(chunk, llm_client)
            processed_sum.append(summary)
            save_md(self.stage, f"summary{i}", summary)
            print(f"took {time.time() - start} seconds")
            start = time.time()
        return processed_sum


In [9]:
# Map files into chunks that can be summarrized 
scout = ScoutAgent("../code_base", model=model_id, debug=True)
files = scout.collect_files()
chunks = scout.create_chunks(files)

In [10]:
len(chunks)

185

In [ ]:
scout.process_all(chunks, client)

Processing chunk 0, with 1 files
took 9.589642763137817 seconds
Processing chunk 1, with 3 files
took 30.69583010673523 seconds
Processing chunk 2, with 1 files
took 30.766769886016846 seconds
Processing chunk 3, with 1 files
took 20.256309032440186 seconds
Processing chunk 4, with 1 files
took 33.97891092300415 seconds
Processing chunk 5, with 3 files
took 31.919766187667847 seconds
Processing chunk 6, with 1 files
took 121.51135683059692 seconds
Processing chunk 7, with 1 files
took 22.47196316719055 seconds
Processing chunk 8, with 1 files
took 35.33953094482422 seconds
Processing chunk 9, with 2 files
took 33.88856101036072 seconds
Processing chunk 10, with 1 files
took 71.34960985183716 seconds
Processing chunk 11, with 1 files
took 18.71541666984558 seconds
Processing chunk 12, with 2 files
took 30.763020038604736 seconds
Processing chunk 13, with 1 files
took 41.48327302932739 seconds
Processing chunk 14, with 1 files
took 19.73542308807373 seconds
Processing chunk 15, with 1 fi

In [11]:
agent_prompt = """Please analyze the following code and provide a summary in Markdown format. 
Include sections for:
1. High-level purpose
2. Key functions/classes
3. Dependencies used"""
formatted_content = f"{agent_prompt}\n\n".join([
    f"--- FILE: {f['path']} ---\n{f['content']}" 
    for f in chunks[0]
])

In [12]:
'''
FOR Ollama server
from openai import OpenAI
client = OpenAI(
    base_url='http://localhost:11434/v1/', 
    api_key='ollama' # Required but ignored locally
)'''

"\nFOR Ollama server\nfrom openai import OpenAI\nclient = OpenAI(\n    base_url='http://localhost:11434/v1/', \n    api_key='ollama' # Required but ignored locally\n)"

In [13]:
system_prompt = """
        You are a system archetect. Give a summary of the given code files.
        Output a Markdown response with a YAML frontmatter header.
        Identify:
        1. Module Name (based on folder or primary file)
        2. Key Dependencies (imports/includes)
        3. Primary Responsibilities
        4. Public APIs or Interfaces used
        """

In [14]:
# NOTE this timed out after 30m with ollama on a 16g M4 and a 8b qwen model after trying on zookeeper chunk 0
response = client.chat.completions.create(
    model=model_id, # or your preferred model
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": formatted_content}
    ]
)

In [15]:
response

ChatCompletion(id='chatcmpl-bef7d7d5-ddeb-4c4d-8f87-3a40b199008e', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content='---\ntitle: Summary of ZooKeeper Code Files\nlayout: markdown\n---\n\n## Module Name\n- **Module Name:** `zookeeper`\n\n## Key Dependencies\n- **Python Standard Library:** `os`, `shutil`, `subprocess`, `sys`, `logging`, `optparse`, `json`, `re`, `time`\n- **Third-party Libraries:** `jira.client`, `requests`, `unittest`, `unittest.mock`\n- **Custom Libraries:** `check_compatibility.py`, `VersionInfoMain.java`, `Info.java`\n\n## Primary Responsibilities\n1. **`zk-merge-pr.py`:**\n   - **Purpose:** Automates the process of merging pull requests from the Apache ZooKeeper GitHub mirror to the Apache ZooKeeper git repo.\n   - **Key Functions/Classes:**\n     - `merge_pr(pr_num, title, pr_repo_desc)`: Merges a pull request into the target branch.\n     - `cherry_pick(pr_num, merge_hash, default_branch)`: Cherry-picks a merge 

In [ ]:
'''
from openai import OpenAI
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI

# Load environment variables
load_dotenv()

def get_model():
    """Returns a local Llama model or a cloud OpenAI model based on config."""
    # OPTIONS: 'local' or 'cloud'
    provider = os.getenv("LLM_PROVIDER", "local").lower()
    
    if provider == "local":
        # Ensure you have run: ollama pull llama3
        return ChatOllama(
            model=MODEL_NAME, 
            base_url="http://localhost:11434",
            temperature=0
        )
    else:
        return ChatOpenAI(
            model="gpt-4o", 
            api_key=os.getenv("OPENAI_API_KEY"),
            temperature=0
        )
'''

In [13]:
task = scout.summarize_chunk(chunk = chunks[0], llm_client = client)

In [22]:
'''# Recommended way to run multiple chunks concurrently
tasks = [ScoutAgent.summarize_chunk(c) for c in chunks]
results = await asyncio.gather(*tasks)'''

'# Recommended way to run multiple chunks concurrently\ntasks = [ScoutAgent.summarize_chunk(c) for c in chunks]\nresults = await asyncio.gather(*tasks)'

In [128]:
chunks[0]

[{'path': 'zookeeper/zk-merge-pr.py',
  'content': '#!/usr/bin/env python\n\n#\n# Licensed to the Apache Software Foundation (ASF) under one or more\n# contributor license agreements.  See the NOTICE file distributed with\n# this work for additional information regarding copyright ownership.\n# The ASF licenses this file to You under the Apache License, Version 2.0\n# (the "License"); you may not use this file except in compliance with\n# the License.  You may obtain a copy of the License at\n#\n#    http://www.apache.org/licenses/LICENSE-2.0\n#\n# Unless required by applicable law or agreed to in writing, software\n# distributed under the License is distributed on an "AS IS" BASIS,\n# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n# See the License for the specific language governing permissions and\n# limitations under the License.\n#\n\n# Utility for creating well-formed pull request merges and pushing them to Apache. This script is a modified version\n# o

In [33]:
class AggregatorAgent:
    def __init__(self, architect_threshold=20000, debug = False, model=MODEL_NAME, stage=STAGES[1]):
        # The character/token limit where we stop summarizing and call the Architect
        self.architect_threshold = architect_threshold
        self.debug = debug
        self.model = model
        self.scout_path = STAGES[1]
        self.stage = stage

    def save_md(self, filename, data):
        try:
            with open(OUTPUT_DIR / self.stage / filename, 'w') as f:
                f.write(data)
        except Exception as e:
            print(f"{e}: data")

    def collect_files(self):
        """Walks the repo and gathers content of relevant files."""
        all_files = []
        for root, _, files in os.walk(OUTPUT_DIR / self.scout_path):
            # Skip hidden folders like .git
            if '/.' in root: continue 

            for file in files:
                full_path = os.path.join(root, file)
                try:
                    with open(full_path, 'r', encoding='utf-8') as f:
                        content = f.read()
                        all_files.append({
                            "path": os.path.relpath(full_path, self.scout_path),
                            "content": content
                        })
                except Exception as e:
                    print(f"Could not read {full_path}: {e}")
        return all_files

    def calculate_total_size(self, summaries):
        return sum(len(s) for s in summaries)

    def summarize_summaries(self, summary_batch, llm_client):
        """
        Collapses a group of summaries into a single 'Module Overview'.
        """
        combined_text = "\n\n".join(summary_batch)
        prompt = (
            "You are a Principal Engineer. Below are several component summaries "
            "from a sub-system. Merge them into a single high-level technical "
            "summary that highlights the internal flow and external dependencies."
        )
        
        response = llm_client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": combined_text}
            ]
        )
        return response.choices[0].message.content

    def reduce(self, summaries, llm_client):
        """
        The recursive loop: Keeps shrinking the data until it fits the threshold.
        """
        current_summaries = summaries
        
        while self.calculate_total_size(current_summaries) > self.architect_threshold:
            print(f"Current size {len(str(current_summaries))} exceeds threshold. Reducing...")
            
            # Split summaries into manageable chunks (e.g., groups of 5)
            new_summaries = []
            chunk_size = 5 
            for i in range(0, len(current_summaries), chunk_size):
                batch = current_summaries[i : i + chunk_size]
                reduced_summary = self.summarize_summaries(batch, llm_client)
                new_summaries.append(reduced_summary)
                self.save_md(f"reduced_sum{i}", reduced_summary)
            
            current_summaries = new_summaries
        if self.calculate_total_size(current_summaries) < self.architect_threshold:
            print(f"Current size {len(str(current_summaries))} Does not need reduction")
            self.save_md(f"reduced_sum", str(current_summaries))
        return current_summaries


In [34]:
# Map files into chunks that can be summarrized 
aggregator = AggregatorAgent(model=model_id, debug=True)
files = aggregator.collect_files()
reduced_files = aggregator.reduce(files, client)

Current size 485579 Does not need reduction


In [36]:
class ArchetectAgent:
    def __init__(self, debug = False, model=MODEL_NAME, stage=STAGES[2]):
        # The character/token limit where we stop summarizing and call the Architect
        self.debug = debug
        self.model = model
        self.agg_path = STAGES[1]
        self.stage = stage

    def save_md(self, filename, data):
        try:
            with open(OUTPUT_DIR / self.stage / filename, 'w') as f:
                f.write(data)
        except Exception as e:
            print(f"{e}: data")

    def collect_files(self):
        """Walks the repo and gathers content of relevant files."""
        all_files = []
        for root, _, files in os.walk(OUTPUT_DIR / self.agg_path):
            # Skip hidden folders like .git
            if '/.' in root: continue 

            for file in files:
                full_path = os.path.join(root, file)
                try:
                    with open(full_path, 'r', encoding='utf-8') as f:
                        content = f.read()
                        all_files.append({
                            "path": os.path.relpath(full_path, self.scout_path),
                            "content": content
                        })
                except Exception as e:
                    print(f"Could not read {full_path}: {e}")
        return all_files

    def calculate_total_size(self, summaries):
        return sum(len(s) for s in summaries)

    def summarize_summaries(self, summary_batch, llm_client):
        """
        Collapses a group of summaries into a single 'Module Overview'.
        """
        combined_text = "\n\n".join(summary_batch)
        prompt = (
            "You are a Senior Architect. Output clean Mermaid TD syntax."
        )
        
        response = llm_client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": combined_text}
            ]
        )
        return response.choices[0].message.content
